# Data Engineer Certification - Practical Exam - Supplement Experiments

1001-Experiments makes personalized supplements tailored to individual health needs.

1001-Experiments aims to enhance personal health by using data from wearable devices and health apps.

This data, combined with user feedback and habits, is used to analyze and refine the effectiveness of the supplements provided to the user through multiple small experiments.

The data engineering team at 1001-Experiments plays a crucial role in ensuring the collected health and activity data from thousands of users is accurately organized and integrated with the data from supplement usage. 

This integration helps 1001-Experiments provide more targeted health and wellness recommendations and improve supplement formulations.


## Task

1001-Experiments currently has the following four datasets with four months of data:
 - "user_health_data.csv" which logs daily health metrics, habits and data from wearable devices,
 - "supplement_usage.csv" which records details on supplement intake per user,
 - "experiments.csv" which contains metadata on experiments, and
 - "user_profiles.csv" which contains demographic and contact information of the users.

Each dataset contains unique identifiers for users and/or their supplement regimen.

The developers and data scientsits currently manage code that cross-references all of these data sources separately, which is cumbersome and error-prone.

Your manager has asked you to write a Python function that cleans and merges these datasets into a single dataset.

The final dataset should provide a comprehensive view of each user's health metrics, supplement usage, and demographic information.

- To test your code, your manager will run only the code `merge_all_data('user_health_data.csv', 'supplement_usage.csv', 'experiments.csv', 'user_profiles.csv')`
- Your `merge_all_data` function must return a DataFrame, with columns as described below.
- All columns must accurately match the descriptions provided below, including names.


## Data

The provided data is structured as follows:

![database schema](schema.png)

The function you write should return data as described below.

There should be a unique row for each daily entry combining health metrics and supplement usage.

Where missing values are permitted, they should be in the default Python format unless stated otherwise.

| Column Name        | Description |
|--------------------|-------------|
| user_id            | Unique identifier for each user. </br>There should not be any missing values. |
| date               | The date the health data was recorded or the supplement was taken, in date format. </br>There should not be any missing values. |
| email              | Contact email of the user. </br>There should not be any missing values. |
| user_age_group  | The age group of the user, one of: 'Under 18', '18-25', '26-35', '36-45', '46-55', '56-65', 'Over 65' or 'Unknown' where the age is missing.|
| experiment_name    | Name of the experiment associated with the supplement usage. </br>Missing values for users that have user health data only is permitted. |
| supplement_name    | The name of the supplement taken on that day. Multiple entries are permitted. </br>Days without supplement intake should be encoded as 'No intake'. |
| dosage_grams       | The dosage of the supplement taken in grams. Where the dosage is recorded in mg it should be converted by division by 1000.</br>Missing values for days without supplement intake are permitted. |
| is_placebo         | Indicator if the supplement was a placebo (true/false). </br>Missing values for days without supplement intake are permitted. |
| average_heart_rate | Average heart rate as recorded by the wearable device. </br>Missing values are permitted. |
| average_glucose    | Average glucose levels as recorded on the wearable device. </br>Missing values are permitted. |
| sleep_hours        | Total sleep in hours for the night preceding the current day’s log. </br>Missing values are permitted. |
| activity_level     | Activity level score between 0-100. </br>Missing values are permitted. |

In [5]:
# Use as many python cells as you wish to write your code

import pandas as pd
import numpy as np

def merge_all_data(health_file, supplement_file, experiments_file, profiles_file):
    """
    Merge all four datasets into a single comprehensive dataset.

    Parameters:
    -----------
    health_file : str
        Path to user_health_data.csv
    supplement_file: str
        Path to supplement_usage.csv
    experiments_file : str
        Path to experiments.csv
    profiles_file : str
        Path to user_profiles.csv

    Returns:
    --------
    pd.DataFrame
        Merged and cleaned dataset with all user health metrics,
        supplement usage, and demographic information.
    """
    # Read all datasets
    health_df = pd.read_csv(health_file)
    supplement_df = pd.read_csv(supplement_file)
    experiments_df = pd.read_csv(experiments_file)
    profiles_df = pd.read_csv(profiles_file)

    # Health Data
    # Clean sleep_hours: remove 'h'/'H' suffix and convert to float
    health_df['sleep_hours'] = health_df['sleep_hours'].str.replace(r'[hH]', '', regex=True).astype(float)

    # Convert date to datetime
    health_df['date'] = pd.to_datetime(health_df['date'])

    # Supplement Data
    # Convert date to datetime
    supplement_df['date'] = pd.to_datetime(supplement_df['date'])

    # Convert dosage form mg to grams (divide by 1000)
    supplement_df['dosage_grams'] = supplement_df['dosage'] / 1000

    # Merge supplement with experiments to get experiment name
    supplement_df = supplement_df.merge(
        experiments_df[['experiment_id', 'name']],
        on='experiment_id',
        how='left'
    )
    supplement_df = supplement_df.rename(columns={'name': 'experiment_name'})

    # User Profiles
    # Define bins and labels for age groups
    bins = [-float('inf'), 17, 25, 35, 45, 55, 65, float('inf')]
    labels = ['Under 18', '18-25', '26-35', '36-45', '46-55', '56-65', 'Over 65']

    profiles_df['user_age_group'] = pd.cut(profiles_df['age'], bins=bins, labels=labels)
    profiles_df['user_age_group'] = profiles_df['user_age_group'].astype(str)
    profiles_df.loc[profiles_df['age'].isna(), 'user_age_group'] = 'Unknown'
    profiles_df['user_age_group'] = profiles_df['user_age_group'].replace('nan', 'Unknown')

    # Merge datasets
    # Outer Merge: Heath and Supplement Data on user_id and date
    merged = health_df.merge(
        supplement_df[['user_id', 'date', 'supplement_name', 'dosage_grams', 'is_placebo', 'experiment_name']],
        on=['user_id', 'date'],
        how='outer'
    )

    # Days w/o supplement intake = 'No intake'
    merged['supplement_name'] = merged['supplement_name'].fillna('No intake')

    # Adding user profiles: email & user_age_group
    merged = merged.merge(
        profiles_df[['user_id', 'email', 'user_age_group']],
        on='user_id',
        how='left'
    )

    # Cobert date to date format (python date objects)
    merged['date'] = merged['date'].dt.date

    # Select and order the final columns as specified
    final_columns = [
        'user_id',
        'date',
        'email',
        'user_age_group',
        'experiment_name',
        'supplement_name',
        'dosage_grams',
        'is_placebo',
        'average_heart_rate',
        'average_glucose',
        'sleep_hours',
        'activity_level'
    ]

    result = merged[final_columns]

    return result

# Test the function
if __name__ == "__main__":
    result = merge_all_data(
        'user_health_data.csv',
        'supplement_usage.csv',
        'experiments.csv',
        'user_profiles.csv'
    )
    print("=== First 5 rows ====")
    print(result.head())
    print(f"\n=== Shape: {result.shape} ===")
    print(f"\n=== Column types ===\n{result.dtypes}")
    print(f"\n=== Date type check ===")
    print(f"\Date type: {type(result['date'].iloc[0])}")
    print(f"\n=== Missing values ===\n{result.isnull().sum()}")
    print(f"\n=== Age group distribution ===\n{result['user_age_group'].value_counts()}")
    print(f"\n=== Supplement name values ===")
    print(f"'No intake' count: {(result['supplement_name'] == 'No intake').sum()}")
    print(f"Other supplements: {result[result['supplement_name'] != 'No intake']['supplement_name'].nunique()} unique")

=== First 5 rows ====
                                user_id  ... activity_level
0  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              1
1  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              3
2  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              1
3  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              1
4  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              1

[5 rows x 12 columns]

=== Shape: (2721, 12) ===

=== Column types ===
user_id                object
date                   object
email                  object
user_age_group         object
experiment_name        object
supplement_name        object
dosage_grams          float64
is_placebo             object
average_heart_rate    float64
average_glucose       float64
sleep_hours           float64
activity_level          int64
dtype: object

=== Date type check ===
\Date type: <class 'datetime.date'>

=== Missing values ===
user_id                 0
date                    0
email                   0
user_age_